In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go


df = pd.read_csv("Cleaned_DF_Hospital_Provider_Cost_Report_2023 (1).csv")

beds_col = "Number of Beds + Total for all Subproviders"
expense_col = "Operating Expense Ratio"
margin_col = "Clinical Operating Margin"
perf_col = "Net Revenue Per Bed"

df = df[[beds_col, expense_col, margin_col, perf_col]].copy()
df[beds_col] = pd.to_numeric(df[beds_col], errors="coerce")
df[expense_col] = pd.to_numeric(df[expense_col], errors="coerce")
df[margin_col] = pd.to_numeric(df[margin_col], errors="coerce")
df[perf_col] = pd.to_numeric(df[perf_col], errors="coerce")
df = df.dropna()

df["Size Group"] = pd.cut(
    df[beds_col],
    bins=[-np.inf, 75, 250, np.inf],
    labels=["Small", "Medium", "Large"],
    right=False
)

df["Expense Group"] = np.where(
    df[expense_col] < 1,
    "Expense Ratio < 1",
    "Expense Ratio ≥ 1"
)

df["Margin Group"] = np.where(
    df[margin_col] >= 0,
    "Positive Margin",
    "Negative Margin"
)

df["Revenue Group"] = pd.qcut(
    df[perf_col],
    q=2,
    labels=["Low Revenue", "High Revenue"],
    duplicates="drop"
)

df = df.dropna(subset=["Size Group", "Expense Group", "Margin Group", "Revenue Group"]).copy()


path_counts = (
    df.groupby(
        ["Size Group", "Expense Group", "MarginGroup" if "MarginGroup" in df.columns else "Margin Group", "Revenue Group"],
        observed=False
    )
    .size()
    .reset_index(name="count")
)

# safety for column naming
if "MarginGroup" in path_counts.columns and "Margin Group" not in path_counts.columns:
    path_counts["Margin Group"] = path_counts["MarginGroup"]

total_hospitals = path_counts["count"].sum()

flow1 = (
    path_counts.groupby(["Size Group", "Expense Group"], observed=False)["count"]
    .sum()
    .reset_index()
)
flow1["share_total"] = flow1["count"] / total_hospitals

flow2 = (
    path_counts.groupby(["Expense Group", "Margin Group"], observed=False)["count"]
    .sum()
    .reset_index()
)
flow2["share_total"] = flow2["count"] / total_hospitals

flow3 = (
    path_counts.groupby(["Margin Group", "Revenue Group"], observed=False)["count"]
    .sum()
    .reset_index()
)
flow3["share_total"] = flow3["count"] / total_hospitals

# -----------------------------
# Nodes
# -----------------------------
labels = [
    "Small", "Medium", "Large",
    "Expense Ratio < 1", "Expense Ratio ≥ 1",
    "Negative Margin", "Positive Margin",
    "Low Revenue", "High Revenue"
]

label_to_index = {label: i for i, label in enumerate(labels)}

node_colors = [
    "#6ad5ff", "#43b7ea", "#1f8fc4",   # size
    "#6ad5ff", "#ff892a",              # expense ratio
    "#A44700", "#ff892a",              # margin
    "#A44700", "#6ad5ff"               # revenue
]

x_positions = [0.04]*3 + [0.30]*2 + [0.60]*2 + [0.96]*2
y_positions = [0.20, 0.50, 0.80] + [0.35, 0.65] + [0.35, 0.65] + [0.35, 0.65]


source, target, value, link_colors, customdata = [], [], [], [], []

# Size -> Expense
for _, row in flow1.iterrows():
    s = str(row["Size Group"])
    e = str(row["Expense Group"])
    source.append(label_to_index[s])
    target.append(label_to_index[e])
    value.append(row["count"])
    link_colors.append("rgba(106,213,255,0.18)")
    customdata.append([s, e, int(row["count"]), row["share_total"]])

# Expense -> Margin
for _, row in flow2.iterrows():
    e = str(row["Expense Group"])
    m = str(row["Margin Group"])
    source.append(label_to_index[e])
    target.append(label_to_index[m])
    value.append(row["count"])
    link_colors.append("rgba(255,137,42,0.22)" if m == "Positive Margin" else "rgba(164,71,0,0.22)")
    customdata.append([e, m, int(row["count"]), row["share_total"]])

# Margin -> Revenue
for _, row in flow3.iterrows():
    m = str(row["Margin Group"])
    r = str(row["Revenue Group"])
    source.append(label_to_index[m])
    target.append(label_to_index[r])
    value.append(row["count"])
    link_colors.append("rgba(164,71,0,0.30)" if r == "Low Revenue" else "rgba(106,213,255,0.22)")
    customdata.append([m, r, int(row["count"]), row["share_total"]])

# -----------------------------
# Plot
# -----------------------------
fig = go.Figure(
    data=[
        go.Sankey(
            arrangement="freeform",   
            node=dict(
                pad=28,
                thickness=22,
                line=dict(color="rgba(90,90,90,0.45)", width=0.8),
                label=labels,
                color=node_colors,
                x=x_positions,
                y=y_positions,
                hovertemplate="<b>%{label}</b><extra></extra>"
            ),
            link=dict(
                source=source,
                target=target,
                value=value,
                color=link_colors,
                customdata=customdata,
                hovercolor="rgba(40,40,40,0.50)",
                hovertemplate=(
                    "<b>%{customdata[0]} → %{customdata[1]}</b><br>"
                    "Hospitals: %{customdata[2]:,}<br>"
                    "Share of all hospitals: %{customdata[3]:.1%}"
                    "<extra></extra>"
                )
            )
        )
    ]
)

fig.update_layout(
    title=None,
    width=1550,
    height=720,
    margin=dict(l=20, r=20, t=140, b=20),

    paper_bgcolor="white",
    plot_bgcolor="white",
    font=dict(size=13, family="Arial", color="#2f2f2f"),

    annotations=[
        dict(x=0.07, y=1.12, xref="paper", yref="paper",
             text="<b>Hospital Size</b>", showarrow=False,
             font=dict(size=15, color="#173a70")),

        dict(x=0.30, y=1.12, xref="paper", yref="paper",
             text="<b>Operating Expense Ratio</b>", showarrow=False,
             font=dict(size=15, color="#173a70")),

        dict(x=0.60, y=1.12, xref="paper", yref="paper",
             text="<b>Clinical Operating Margin</b>", showarrow=False,
             font=dict(size=15, color="#173a70")),

        dict(x=0.93, y=1.12, xref="paper", yref="paper",
             text="<b>Revenue Performance</b>", showarrow=False,
             font=dict(size=15, color="#173a70"))
    ]
)

fig.show(config={
    "displayModeBar": False,
    "scrollZoom": False,
    "editable": False
})